In [ ]:
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

sns.set_palette("viridis")


In [ ]:
data_train: pd.DataFrame = pd.read_csv("network-intrusion-detection/versions/1/Train_data.csv")
data_test: pd.DataFrame = pd.read_csv("network-intrusion-detection/versions/1/Test_data.csv")
data_train.info()

In [ ]:
data_train.head()

In [ ]:
print("Training data NANs:\n",data_train.isna().sum(), "\n\n")
print("Testing data NANs:\n",data_test.isna().sum())
# Which should all be 0.

In [ ]:
#plotting the output class distributions
sns.countplot(x='class' , data = data_train)
plt.title("Output Distribution")

In [ ]:
from sklearn.preprocessing import LabelEncoder
# Giving labels to variables that contain objects.
objects_train = data_train.select_dtypes(include=['object']).columns
objects_test = data_test.select_dtypes(include=['object']).columns

# Initializing the LabelEncoder 
label_encoder = LabelEncoder()
# Applying a new label to each object collumn
for col in objects_train:
    data_train[col] = label_encoder.fit_transform(data_train[col]) #type: ignore

for col in objects_test:
    data_test[col] = label_encoder.fit_transform(data_test[col])#type: ignore
    
data_train.head()


In [ ]:
from sklearn.model_selection import train_test_split
X = data_train.drop("class", axis=1)
y = data_train["class"]
if "class" in X: # To ensure proper training
    raise Exception("X contains output class")
X_test = data_test

X_train , X_val , y_train , y_val = train_test_split(X, y, test_size=0.2 , random_state=42)
print(X_train.shape)
print(X_val.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=1000, random_state=42)

# Fitting the classifier on the training data
rf.fit(X_train , y_train)


In [ ]:
from sklearn.metrics import accuracy_score
# Predicting the output variable
y_pred = rf.predict(X_val)

print("Accuracy score: ", accuracy_score(y_val, y_pred))

In [ ]:
import shap as SHAP

# SHAP explanation variables
explainer = SHAP.KernelExplainer(rf.predict, SHAP.kmeans(X_train, 10))
nb_points_explain = round(0.2*X_train.shape[0])
shap_values = explainer(X_train.iloc[0:nb_points_explain, :])